In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

import plotly.express as px

import scanpy as sc
import scipy.sparse as sp

import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm

def plot_sample(df, id):
    current_df = df.loc[id]
    current_df = current_df[current_df != 0].sparse.to_dense()
    n_movies = len(current_df)

    bin_edges = np.arange(0.5, 11.5, 1)

    plt.figure(figsize=(3, 2))
    plt.hist(current_df.values.flatten(), bins=bin_edges, color='skyblue', edgecolor='black')
    plt.xticks(np.arange(1, 11))
    plt.title(f'{id} - {n_movies} ratings')
    plt.xlim([0.5,10.5])
    plt.show()

In [3]:
users_df = pd.read_pickle('users_df_complete.pkl')

In [4]:
users_df.index

Index(['hexagore', 'chrissytopher', 'nolan380', 'lovxseamon', 'marc3lis',
       'guiswhoi', 'iwishiwasafinch', 'cesarchagas', 'l1cheesweetpea',
       'starbells',
       ...
       'moviemaxwell', 'cocoapowder', 'motionlmags', 'marian64', 'mjc0111',
       'moviebiz', 'theburkenation', 'florences_oscar', 'tediously_brief',
       'falonwillow'],
      dtype='object', length=1856)

In [ ]:
user = 'cocoapowder'

selected_user = users_df.loc[['cocoapowder']]
selected_user = selected_user.loc[:, (selected_user != 0).any(axis=0)]

Index(['weapons-2025', 'the-naked-gun', 'i-know-what-you-did-last-summer-2025',
       'm3gan-20', 'fear-street-prom-queen', 'dangerous-animals',
       'bring-her-back', 'until-dawn-2025', 'sinners-2025', 'warfare',
       ...
       'our-loved-ones', 'lucy-grizzli-sophie', 'two-days-before-christmas',
       'happy-days-2000', 'madchen-madchen', '1995-2024',
       'sweet-summer-pow-wow', 'denial-is-a-river', 'madame-de-sevigne',
       'sophie-lavoie'],
      dtype='object', length=898)

In [43]:
user = 'cocoapowder'


users_df = users_df.replace(0, np.nan)

target = users_df.loc[user].to_numpy()


cos_sims = []
for u, row in users_df.iterrows():
    row_arr = row.to_numpy()

    # mask NaNs (only keep overlapping indices)
    mask = ~np.isnan(target) & ~np.isnan(row_arr)
    if mask.sum() == 0:
        cos_sim = np.nan
    else:
        dot = np.dot(target[mask]-5.5, row_arr[mask]-5.5)
        cos_sim = dot / (np.linalg.norm(target[mask]-5.5) * np.linalg.norm(row_arr[mask]-5.5))
    
    cos_sims.append((u, cos_sim))

similarity_df = pd.DataFrame(cos_sims, columns=['user', 'cos_sim'])

In [44]:
print(similarity_df.sort_values(by='cos_sim', ascending=False))
similarity_df.sort_values(by='cos_sim', ascending=False).to_csv('test_similarity.csv')

                 user   cos_sim
1847      cocoapowder  1.000000
10    clementineultra  0.986440
1       chrissytopher  0.953624
755           kristen  0.936111
536              mskp  0.930626
...               ...       ...
1492          bennash -0.148189
1327    margotwendice -0.207357
843          max_ahoy -0.219465
777      supremelemon -0.744063
1136             tmgg       NaN

[1856 rows x 2 columns]


In [ ]:
#similarity_df['cos_sim'] = 1

In [45]:
similarity_df.fillna(0)

others_df = users_df.drop(index=user)

# Align similarities with others
weights = similarity_df.set_index('user').loc[others_df.index, 'cos_sim']

# Compute weighted average per column (movies)
# (row-wise multiply each user’s ratings by its similarity weight)
weighted_sum = (others_df.T * weights).T.sum(axis=0)
sum_weights = weights.sum()

weighted_avg = weighted_sum / sum_weights

In [46]:
weighted_avg.sort_values(ascending=False)

parasite-2019                        8.765893
spider-man-into-the-spider-verse     8.071592
oppenheimer-2023                     7.986204
everything-everywhere-all-at-once    7.979747
whiplash-2014                        7.930428
                                       ...   
our-twisted-hero                    -0.004622
jealousy-is-my-middle-name          -0.004956
my-memories-of-old-beijing          -0.005207
deep-blue-night                     -0.005313
the-people-in-white                 -0.005778
Length: 217142, dtype: float64

In [47]:
mask_unrated = users_df.loc[user].isna()
recommendations = weighted_avg[mask_unrated]

In [ ]:
recommendations.sort_values(ascending=False)

parasite-2019                        8.765893
spider-man-into-the-spider-verse     8.071592
oppenheimer-2023                     7.986204
everything-everywhere-all-at-once    7.979747
whiplash-2014                        7.930428
                                       ...   
our-twisted-hero                    -0.004622
jealousy-is-my-middle-name          -0.004956
my-memories-of-old-beijing          -0.005207
deep-blue-night                     -0.005313
the-people-in-white                 -0.005778
Length: 217142, dtype: float64

In [13]:
user = 'cocoapowder'

similarity_movies = selected_user.columns

total_users = users_df.index

similarity_df = pd.DataFrame()
similarity_df['user'] = total_users
similarity_df['cos_sim'] = 0

for u in tqdm(total_users):
    current_df = users_df.loc[[user, u],:]
    current_df = current_df.loc[:, (current_df != 0).all(axis=0)]

    print(current_df.shape)

    row1 = current_df.iloc[0].to_numpy()
    row2 = current_df.iloc[1].to_numpy()

    cos_sim = np.dot(row1, row2) / (np.linalg.norm(row1) * np.linalg.norm(row2))

    similarity_df.loc[u, 'cos_sim'] = cos_sim

  0%|          | 1/1856 [00:10<5:14:33, 10.17s/it]

(2, 205)


  0%|          | 2/1856 [00:19<4:57:47,  9.64s/it]

(2, 87)


  0%|          | 3/1856 [00:29<5:03:43,  9.83s/it]

...

(2, 295)


  2%|▏         | 30/1856 [05:22<7:50:53, 15.47s/it]

(2, 342)


  2%|▏         | 30/1856 [05:34<5:39:35, 11.16s/it]


KeyboardInterrupt: 